In [30]:
%pip install langchain_openai
%pip install langchain
%pip install json
%pip install typing

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement json (from versions: none)
ERROR: No matching distribution found for json


Note: you may need to restart the kernel to use updated packages.


### RAG Retriever with Caller Provided Question & Context (w/o VecDB)

In [31]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from typing import List, Dict

PROMPT_TEMPLATE = """
You are a digital marketing agent for La Roche-Posay skincare.

Use the product descriptions provided below and the customer profile in the question to write 3 short, creative, and engaging social media advertisements (under 280 characters each). Tailor the tone and content to the user’s needs. Do not repeat the same format. Be persuasive and helpful.

{context}

Customer Profile: {question}

3 Personalized Ad Variations:
"""


def create_retriever_wo_vecdb():
    llm = ChatOpenAI(
        model_name="gpt-4o",
        openai_api_key="Enter your key",  # Replace with valid key
        temperature=0.7
    )
    
    prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
    
    def query(question: str, docs: list[str]) -> Dict:
        """Execute full Naïve RAG pipeline to generate ad copy"""
        # Combine context
        context = "\n\n".join(docs)
        
        # Generate answer
        response = llm.invoke(prompt.format_messages(
            context=context,
            question=question
        ))
        
        return {
            "answer": response.content,
            "sources": [{"content": d} for d in docs]
        }
    
    return query

### Load Relevant Docs (Will be Replaced by VecDB)

In [32]:
import json
import os

def load_relevant_docs_json():
    filenames = [
        "Anthelios_doc_1.json",
        "Effaclar_doc_2.json",
        "Retinol_B3_doc_3.json",
        "Toleriane_doc_4.json"
    ]
    return [json.load(open(os.path.join("jsons", fname)))["page_content"] for fname in filenames]


def load_relevant_docs_text():
    filenames = [
        "Anthelios_doc_1.txt",
        "Effaclar_doc_2.txt",
        "Retinol_B3_doc_3.txt",
        "Toleriane_doc_4.txt"
    ]
    return [open(os.path.join("texts", fname), encoding="utf-8").read().strip() for fname in filenames]

# Usage:
# docs = load_relevant_docs_json()
# OR
# docs = load_relevant_docs_text()


### Call the RAG retriever without VecDB (User provides context & query)

In [33]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "true"

# Updated loader: uses actual filenames
def load_relevant_docs_text():
    filenames = [
        "Anthelios_doc_1.txt",
        "Effaclar_doc_2.txt",
        "Retinol_B3_doc_3.txt",
        "Toleriane_doc_4.txt"
    ]
    return [open(os.path.join("texts", fname), encoding="utf-8").read().strip() for fname in filenames]

# Load the RAG system
rag_system = create_retriever_wo_vecdb()

# Sample customer profile (this is the query)
query = "Man in his 30s starting skincare"

# Load product descriptions
docs = load_relevant_docs_text()

# Run RAG system
result = rag_system(question=query, docs=docs)

# Output generated ads
print(result['answer'])


1. 🌞 Say goodbye to sun damage! Our award-winning sunscreen with Cell-Ox Shield® tech offers top-tier protection in a lightweight, matte finish. Perfect for your daily routine. Start strong, stay protected. #LaRochePosay

2. 🧴 New to skincare? Effaclar Salicylic Acne Treatment is your go-to for tackling stubborn imperfections. With 0.5% Salicylic Acid, it hydrates while fighting acne. Perfect for a fresh start! #ClearSkinGoals

3. ✨ Elevate your routine with our anti-aging retinol serum. Smoother, hydrated skin is just a step away. Ideal for sensitive skin, it reduces lines and sun damage. Youthful skin awaits! #TimelessGlow
